# BDD100K — Task 2: Model

**Bosch Applied CV Assignment | Notebook 02**

This notebook covers:
1. Model selection rationale
2. YOLOv8 architecture explanation
3. Loading the pre-trained YOLOv8n model
4. Running inference on BDD100K validation images
5. Custom PyTorch DataLoader for BDD100K
6. Training pipeline demonstration (1 epoch on 1000-image subset)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

from src.model.model_loader import YOLOv8Detector, BDD_CLASSES
from src.model.dataloader import BDD100KDetectionDataset, BDD100KDataModule
from src.data_analysis.bdd_parser import BDDDataset

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

## 1. Model Selection Rationale

### Chosen Model: YOLOv8n (Ultralytics, 2023)

| Criterion | YOLOv8n | RT-DETR | Faster R-CNN |
|-----------|---------|---------|-------------|
| Speed (FPS on GPU) | ~120+ | ~60 | ~20 |
| mAP on COCO | 37.3 | 53.1 | 37.0 |
| Real-time capable | ✅ | ✅ | ❌ |
| BDD class overlap | High | High | High |
| Ease of fine-tuning | ✅ | Moderate | Moderate |
| Export (ONNX/TRT) | ✅ | ✅ | Limited |

**Why YOLOv8n specifically:**
- BDD100K is an **autonomous driving** dataset — real-time inference is non-negotiable
- All 10 BDD classes map to COCO classes (pre-trained weights transfer directly)
- The nano variant (3.2M params) is efficient enough to deploy on embedded hardware (Jetson)
- Anchor-free detection with DFL loss handles small objects (cyclists, pedestrians) better than YOLOv5

### Architecture Summary
```
Input [640×640×3]
    │
    ▼
Backbone: CSPDarknet + C2f modules
  ├── C2f-1  [320×320×64]   ← Stem
  ├── C2f-2  [160×160×128]  ← Stage 2
  ├── C2f-3  [80×80×256]    ← Stage 3 (P3)
  ├── C2f-4  [40×40×512]    ← Stage 4 (P4)
  └── SPPF   [20×20×512]    ← Stage 5 (P5, with SPP pooling)
    │
    ▼
Neck: PANet (top-down + bottom-up feature pyramid)
  ├── P3 feature map [80×80]
  ├── P4 feature map [40×40]
  └── P5 feature map [20×20]
    │
    ▼
Head: Decoupled detection head (anchor-free)
  ├── Classification branch (BCE loss)
  └── Regression branch (CIoU + DFL loss)
    │
    ▼
Output: Bounding boxes + class probabilities
```

## 2. Load Pre-trained YOLOv8n Model

In [ ]:
# Initialize and load the detector
detector = YOLOv8Detector(
    model_name='yolov8n.pt',  # Downloads automatically from Ultralytics if not cached
    conf_threshold=0.25,
    iou_threshold=0.45,
    device='cpu',  # Change to 'cuda' if GPU available
)
detector.load()

In [ ]:
# Inspect model architecture
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
print(f'Model parameters: {sum(p.numel() for p in model.model.parameters()):,}')
print(f'COCO classes: {len(model.names)}')
print('\nFirst 15 COCO classes:')
for i in range(15):
    print(f'  {i}: {model.names[i]}')

## 3. Inference on a Sample BDD100K Image

In [ ]:
# ── Update this path to a real BDD100K validation image ──
SAMPLE_IMAGE = '../Data/images/val/b1c9c847-3bda4659.jpg'

if not os.path.exists(SAMPLE_IMAGE):
    print('Image not found. Please update SAMPLE_IMAGE path to a real BDD100K image.')
    print('Skipping inference demo — set SAMPLE_IMAGE to a valid path.')
else:
    detections, annotated = detector.predict_image(SAMPLE_IMAGE, return_annotated=True)

    print(f'Found {len(detections)} objects:')
    for det in detections:
        print(f'  {det.category:<18} conf={det.confidence:.3f}  box=({det.x1:.0f},{det.y1:.0f},{det.x2:.0f},{det.y2:.0f})')

    if annotated is not None:
        plt.figure(figsize=(16, 8))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title('YOLOv8n Inference on BDD100K Validation Image')
        plt.show()

## 4. Custom PyTorch DataLoader for BDD100K

In [ ]:
TRAIN_JSON = '../Data/labels/bdd100k_labels_images_train.json'
VAL_JSON   = '../Data/labels/bdd100k_labels_images_val.json'
TRAIN_IMAGE_DIR = '../Data/images/train'
VAL_IMAGE_DIR   = '../Data/images/val'

# Initialize the data module
data_module = BDD100KDataModule(
    train_json=TRAIN_JSON,
    val_json=VAL_JSON,
    train_image_dir=TRAIN_IMAGE_DIR,
    val_image_dir=VAL_IMAGE_DIR,
    batch_size=4,
    img_size=640,
    num_workers=0,  # Set to 4+ on Linux/Mac
    subset_size=100,  # Small subset for demo
)
train_loader, val_loader = data_module.build()

In [ ]:
# Inspect a batch
import torch

for images, targets in train_loader:
    print(f'Images batch shape : {images.shape}')    # (B, 3, 640, 640)
    print(f'Targets shape      : {targets.shape}')   # (N_total_objects, 6)
    print(f'Image min/max      : {images.min():.3f} / {images.max():.3f}')
    print(f'Target columns     : [batch_idx, class_id, cx_norm, cy_norm, w_norm, h_norm]')
    print(f'Sample target rows :')
    print(targets[:5])
    break

In [ ]:
# Visualize a batch of training images with ground truth boxes
from src.data_analysis.bdd_parser import BDD_DETECTION_CLASSES

CLASS_COLORS = plt.cm.tab10(np.linspace(0, 1, 10))

images, targets = next(iter(train_loader))
B = images.shape[0]

fig, axes = plt.subplots(1, B, figsize=(5 * B, 5))
if B == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    img = images[i].permute(1, 2, 0).numpy()
    ax.imshow(img)
    
    # Draw GT boxes for this image
    img_targets = targets[targets[:, 0] == i]
    H, W = img.shape[:2]
    
    for t in img_targets:
        cls_id = int(t[1])
        cx, cy, w, h = t[2].item(), t[3].item(), t[4].item(), t[5].item()
        x1 = (cx - w/2) * W
        y1 = (cy - h/2) * H
        x2 = (cx + w/2) * W
        y2 = (cy + h/2) * H
        color = CLASS_COLORS[cls_id % 10]
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                              linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 3, BDD_DETECTION_CLASSES[cls_id],
                color=color, fontsize=8, fontweight='bold')
    
    ax.set_title(f'Image {i} — {len(img_targets)} objects')
    ax.axis('off')

plt.suptitle('BDD100K Training Batch (Custom DataLoader)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Training Pipeline — 1 Epoch on Subset (Bonus Points)

In [ ]:
from src.model.trainer import BDDYOLOTrainer

trainer = BDDYOLOTrainer(
    train_json=TRAIN_JSON,
    val_json=VAL_JSON,
    train_image_dir=TRAIN_IMAGE_DIR,
    val_image_dir=VAL_IMAGE_DIR,
    output_dir='../results/training',
    model_name='yolov8n.pt',
    subset_size=1000,    # 1000 images for the demo
    epochs=1,
    img_size=640,
    batch_size=8,
    device='cpu',        # Change to 'cuda' for GPU
)

In [ ]:
# Step 1: Convert BDD100K annotations to YOLO format
data_yaml = trainer.prepare_dataset()
print(f'YOLO dataset ready: {data_yaml}')

In [ ]:
# Step 2: Run training (1 epoch on 1000-image subset)
# WARNING: This will take several minutes on CPU; use GPU for reasonable speed
# Uncomment to run:

# best_model = trainer.train()
# print(f'Best model: {best_model}')

---
## Summary

- **Model**: YOLOv8n pre-trained on COCO, mapping COCO→BDD100K classes
- **DataLoader**: Custom `BDD100KDetectionDataset` with YOLO-format normalized labels
- **Training**: `BDDYOLOTrainer` converts BDD JSON → YOLO format → calls `yolo.train()`
- **Key strength**: Pre-trained COCO weights transfer well to BDD100K due to class overlap

➡ See **Notebook 03** for evaluation and visualization of results.